In [ ]:
# GUMPLE port of MCDataComparisonPID-reCAF.ipynb (sbn-rewgted-19 dataframes).
# cwd guard + sys.path setup for headless nbconvert (kernel starts in nb/).
import os, sys
if os.path.basename(os.getcwd()) == "nb":
    os.chdir("..")
# mirror the sys.path setup of the gump scripts: the gump dir (kinematics,
# loaddf), ../gumple (gumple_cuts) and the repo root (pyanalib, makedf) --
# headless
# nbconvert has no PYTHONPATH from setup.sh
for _p in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), "..", "..")),
           os.path.abspath(os.path.join(os.getcwd(), "..", "gumple"))):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("cwd:", os.getcwd())

# MC-Data Comparison of PID Variables vs. the Calorimetric Systematic Model

MC-data comparisons of the four chi2 PID variables, broken down by true particle
type, under one or more calorimetric/dE/dx-smearing systematic models
(`SMEAR_MODELS`). Each model varies which calorimetric terms (and at what size)
enter the systematic error band; the MC central value stays nominal. Plots are
made at each cut stage in `CUT_STAGES` and saved with a per-stage, per-model
filename tag.

**Cut stages.** Three nested stages, each adding to the one above it, so comparing
them isolates what each group of cuts does to the chi2 distributions:

| # | filename tag | cut |
|---|---|---|
| 1 | `simplecosrej` | `FV` & `nu_score > 0.6` & `n_pfp == 2` |
| 2 | `simplecosrej_trkqual_nop` | + `mu_trackScore > 0.6` & `mu_len > 50` cm |
| 3 | `simplecosrej_trkqual_p3` | + `p_len > 3` cm |

`FV` is the `FV()` callable (also the load-time preselection):
`gc.sanity_cut` (slice vertex non-nan) & `gc.slcfv_cut` (vertex 10 cm in from x/y
and the front of z, 50 cm from the back of z, per detector and run, minus the
ICARUS WW dangling-cable region) & `cut_contained` (production flag: every pfp
starts and ends in the vertex's TPC and is contained by 10 cm) & `cut_cathode`
(production flag: no pfp crosses the cathode -- SBND only, `True` for ICARUS).
The flash cut (`gc.flash_cut`: `flash_maxpe` > 5000 PE ICARUS Run2, > 1000 Run4,
> 2000 SBND) is AND-ed onto **every** stage; it sits outside `FV()` only so the
trigger systematic can move its threshold in both directions.

Stage 2 puts **no cut at all on the proton candidate**, so stage 2 vs. stage 3
brackets the short-proton veto: whatever differs between them is exactly what the
`p_len > 3` cm requirement removes. (An earlier revision instead compared 5 cm and
3 cm working points; the 5 cm stage is dropped.)

**Chi2 flavor (`CHI2_FLAVORS`, `GUMPLE_CHI2_FLAVOR`)**: which chi2 calculation is
plotted. *Which flavors exist depends on the production* -- `sbn-rewgted-21`
stores one (`nominal`, the angle-blended default PID), `sbn-rewgted-20-calovarB`
stores four (the untrimmed collection plane plus three end-of-track trims). One
run of this notebook plots exactly one flavor, into its own `pid/<flavor>/`
sub-directory. See the RUN CONFIGURATION cell for the full story, including why
the `nominal` flavor is labelled "Default PID" rather than "Coll. Plane".

**Calorimetric model set (`GUMPLE_CALO_MODEL`)**: `scan` (the default) runs every
model in the scan -- 10 for ICARUS, 11 for SBND -- so the size of each term can be
compared. `nominal` runs the single tuned per-detector budget instead: ICARUS 1%
gain + dE/dx constant-resolution smearing + dE/dx bias; SBND 2% gain + 2x
constant-resolution smearing + EMB.

**Angular binning (`GUMPLE_ANGLE_BINS`)**: on by default, adding five
equal-statistics bins of each of two track angles to the inclusive comparison.
Set to `0` for a sample too small to support the split.

In [ ]:
%load_ext autoreload
%autoreload 2

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib as mpl

from tqdm.auto import tqdm

import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

from multiprocess import Pool

from scipy import stats


In [ ]:
import pyanalib.pandas_helpers as ph
from makedf.util import *

import kinematics
import gumple_cuts as gc
import loaddf
import syst
import importlib

# loaddf pulls in rwt_map -> PID -> plot_tools, which applies a global mplstyle
# as an import side effect (plot_tools.py:27). Every style property these plots
# care about is set explicitly below, so reset to the matplotlib defaults and
# keep this series looking like the earlier ones.
mpl.rcdefaults()


In [ ]:
# ============================================================
# RUN CONFIGURATION
# ============================================================
# Parameterized by environment variable for the serial runner
# (run_gumple_serial.py), so one notebook serves every detector x chi2-flavor
# combination under headless nbconvert:
#   GUMPLE_DETECTOR    -- "SBND", "ICARUS Run2", "ICARUS Run4"
#   GUMPLE_CHI2_FLAVOR -- which chi2 PID calculation to plot (see CHI2_FLAVORS)
#   GUMPLE_CALO_MODEL  -- "scan" (default: the full model scan) or "nominal"
#                         (the single tuned per-detector budget). See
#                         build_smear_models below.
#   GUMPLE_ANGLE_BINS  -- "1" (default) to bin in angular quintiles as well as
#                         inclusively, "0" for the inclusive stage only
#   GUMPLE_DF_DIR      -- input production directory   (cell below)
#   GUMPLE_PLOTBASE    -- output plot base directory   (cell below)
#   GUMPLE_NMC         -- cap on the number of CV MC files (cell below)
DETECTOR = os.environ.get("GUMPLE_DETECTOR", "ICARUS Run2")
INCLUDE_DIRT = True

# Angular quintile binning. Every comparison is made inclusively and, with this
# on, in five equal-statistics bins of each of two track angles as well (see the
# ANGULAR BINNING cell). Turning it off leaves the inclusive stage only, which
# is all a low-statistics sample can support -- ICARUS Run 4 rides the prescaled
# "unblind" stream, since this production has no Run 4 FullOnBeam file.
DO_ANGLE_BINS = os.environ.get("GUMPLE_ANGLE_BINS", "1") not in ("0", "", "false", "False")

# ============================================================
# CHI2 PID FLAVORS
# ============================================================
# Which hits of the calorimetry the chi2 PID is computed from. The columns are
#     {mu,prot}_chi2<plane><variation>_of_{mu,prot}_cand
# with <plane> naming the calculation and <variation> either the nominal ("" for
# the default calculation, "cafpyana" for the trims) or one of the calo/smear
# systematic variations below.
#
# WHICH FLAVORS EXIST IS PRODUCTION-DEPENDENT:
#
#   sbn-rewgted-21 stores ONE calculation -- 70 chi2 columns, the bare nominal
#   plus the 16 calorimetric variations. The alternates were turned off in
#   production (maple/makedf.py do_alt_chi2 / do_ind_chi2 / do_cafana_chi2), so
#   none of the p2trim*_ / p0_ / p1_ / bestplane_ / cafana columns exist. Use
#   the "nominal" flavor there.
#
#   That surviving nominal is ANGLE-BLENDED on the proton candidate
#   (maple/makedf.py PROT_TRIM_ANGLE_X_DEG / DEFAULT_PROT_TRIM_TAG): the
#   untrimmed collection plane for theta_x > 47 deg, and the last-2cm-trimmed
#   collection plane (p2trim2) for candidates nearly along the drift, where the
#   reconstructed track end the collection-plane chi2 samples heavily is most
#   distorted. The MUON candidate always uses the untrimmed collection plane.
#   Hence the "Default PID" label rather than "Coll. Plane": only the
#   *_of_mu_cand variables are purely the collection plane.
#
#   NB the thetadrift quintile edges below straddle that 47 deg boundary almost
#   exactly (the lowest quintile ends at 46.9 deg SBND / 46.6 deg ICARUS Run 2),
#   so the lowest thetadrift bin is close to "the trimmed calculation" and the
#   four above it to "the untrimmed calculation". That is a useful diagnostic of
#   the blend, not an artifact to correct for.
#
#   sbn-rewgted-20-calovarB stores four flavors -- 274 chi2 columns: the full
#   collection-plane hit set plus three progressively more aggressive end-of-
#   track trims, each carrying the full cafpyana calo/smear variation set. The
#   trims are ordered by how much they remove: on ICARUS Run2 CV MC the median
#   mu_chi2_of_mu_cand falls 7.93 (untrimmed) -> 7.36 -> 7.16 -> 6.81 and the
#   fraction with no chi2 at all rises 0.7% -> 0.8% -> 0.9% -> 1.0%.
#
#   The earlier -calovar production had a DIFFERENT set again: the three wire
#   planes plus a best-plane pick (p0_/p1_/bestplane_) and a "cafana"
#   CAFANA-compat algorithm on the CAF-stored dE/dx. None of those exist in
#   either of the productions above.
#
# One run of this notebook plots exactly one flavor, into its own pid/<flavor>/
# sub-directory.
#
# Entries: tag -> (column plane prefix, nominal variation suffix, plot label)
CHI2_FLAVORS = {
    "nominal": ("",          "",         "Default PID"),
    "p2":      ("",          "",         "Coll. Plane"),
    "p2trim":  ("p2trim_",   "cafpyana", "Coll. Plane, Trim 1"),
    "p2trim2": ("p2trim2_",  "cafpyana", "Coll. Plane, Trim 2"),
    "p2trim3": ("p2trim3_",  "cafpyana", "Coll. Plane, Trim 3"),
}

CHI2_FLAVOR = os.environ.get("GUMPLE_CHI2_FLAVOR", "p2")
if CHI2_FLAVOR not in CHI2_FLAVORS:
    raise ValueError("unknown GUMPLE_CHI2_FLAVOR %r (choose from %s)"
                     % (CHI2_FLAVOR, sorted(CHI2_FLAVORS)))
CHI2_PLANE, CHI2_NOMVAR, CHI2_LABEL = CHI2_FLAVORS[CHI2_FLAVOR]

# The four per-slice chi2 variables, in their NOMINAL (flavor-free) names. Every
# frame is rewritten by apply_flavor() so that these names hold the selected
# flavor's values -- the plotting, cut and systematic machinery below then needs
# no flavor awareness at all.
chi2vars = [
    "mu_chi2_of_mu_cand",
    "prot_chi2_of_mu_cand",
    "mu_chi2_of_prot_cand",
    "prot_chi2_of_prot_cand",
]

def chi2_col(par, cand, var=""):
    """Stored column name of one chi2 variable of the selected flavor.

    par/cand are "mu" or "prot"; var is "" (nominal, default calculation only),
    "cafpyana", or a calo/smear variation suffix."""
    return "%s_chi2%s%s_of_%s_cand" % (par, CHI2_PLANE, var, cand)

def apply_flavor(frame):
    """In place: point the four nominal chi2 columns at the selected flavor.

    A no-op for the flavors whose nominal IS the bare column ("nominal", "p2").
    EVERY frame that is histogrammed or that feeds a systematic has to go through
    this (CV MC, dirt, detector variations, on/off-beam data); frames derived
    later from those (df_ooav, be_df, the track-split frames) inherit it.

    Assignment is by numpy array (positional) because the CV frames carry
    duplicate index labels."""
    if CHI2_PLANE == "" and CHI2_NOMVAR == "":
        return frame
    for par in ("mu", "prot"):
        for cand in ("mu", "prot"):
            frame["%s_chi2_of_%s_cand" % (par, cand)] = \
                frame[chi2_col(par, cand, CHI2_NOMVAR)].to_numpy()
    return frame

# The dE/dx bias variation is an ICARUS-only production. The mu/prot_chi2dedxbias_*
# columns do exist in the SBND CV files, but the variation was not generated there
# (maple/makedf.py _add_dedx_variations sets it equal to CV on SBND), so they must
# not be used as a systematic universe for SBND.
HAS_DEDXBIAS = "ICARUS" in DETECTOR

# The EMB R+0.25 recombination variation (mu/prot_chi2R_p25_* columns) is
# SBND-only in the same sense, and is scanned as a systematic model there.
HAS_RP25 = DETECTOR == "SBND"

# ============================================================
# SMEARING / CALORIMETRIC SYSTEMATIC MODELS
# ============================================================
# Each model is a set of dE/dx-smearing terms added to the systematic error
# budget, plus the list of base calo variations kept in that model's budget, plus
# a scale factor on the GEANT4 (re-interaction) budget. The MC central value
# stays nominal in every model.
#
# Terms enter the covariance independently (in quadrature); a norm of N scales a
# term's covariance by N^2, so ("smear13", 2) is the smear13 variation at twice
# its nominal size. g4_norm does the same for the whole G4 term.
#
# Model entries: (tag-for-filename, label-drawn-on-plot, [(variation, norm), ...],
#                 [base calo variations], g4_norm)
#
# The calo variation names and their sizes come from maple/makedf.py
# _add_dedx_variations:
#   "hi"        gain scaled up 1 sigma -- ICARUS 1% (1.01), SBND 2% (1.02)
#   "smear13"   dE/dx constant resolution, 13%
#   "sqsmear15" dE/dx stochastic (sqrt) smearing, 15%
#   "dedxbias"  dE/dx spline bias, ICARUS only
#   EMB (effective modified box) recombination parameters, from chi2pid.py:
#   "alpha_p" (0.904 + 0.008), "beta_p" (0.204 + 0.008), "R_p" (1.25 + 0.02),
#   "R_p25"   (1.25 + 0.25, an SBND-only large-R test)
EMB_VARIATIONS = ["alpha_p", "beta_p", "R_p"]  # EMB recombination parameters
G4_NORM_SCALE = 2      # combined_g4x2's G4 budget scale
EMB_R = "R_p"          # embR10_smear13's scaled EMB term
EMB_R_NORM = 10

# THE NOMINAL (TUNED) CALORIMETRIC BUDGET, one model per detector. This is the
# budget the signal box uses -- mcdata_comparison_gumple.py CHI2_VARIATIONS /
# CHI2_NORM -- with the ICARUS dE/dx bias term added:
#   ICARUS  1% gain, dE/dx constant-resolution smearing, dE/dx bias. No EMB.
#   SBND    2% gain, dE/dx constant-resolution smearing at 2x, EMB.
# The asymmetry is a choice, not a data limitation, except for dE/dx bias, which
# genuinely does not exist for SBND (HAS_DEDXBIAS). Neither detector takes
# sqsmear15.
NOMINAL_CALO_MODEL = {
    "ICARUS": ("nominal", "Nominal Calo. Syst.",
               [("smear13", 1), ("dedxbias", 1)], ["hi"],                     1),
    "SBND":   ("nominal", "Nominal Calo. Syst.",
               [("smear13", 2)],                  ["hi"] + EMB_VARIATIONS,    1),
}


def build_smear_models(detector, model_set):
    """The list of smearing/calo systematic models for one detector.

    model_set "nominal" gives the single tuned per-detector budget above;
    "scan" gives the full scan (10 models for ICARUS, 11 for SBND)."""
    has_dedxbias = "ICARUS" in detector
    has_rp25 = detector == "SBND"

    if model_set == "nominal":
        return [NOMINAL_CALO_MODEL["SBND" if detector == "SBND" else "ICARUS"]]
    if model_set != "scan":
        raise ValueError("unknown GUMPLE_CALO_MODEL %r (scan or nominal)" % model_set)

    models = [
        ("nosmear",             "No dE/dx Smearing",                   [],                                 ["hi"] + EMB_VARIATIONS, 1),
        ("smear13",             "dE/dx Constant Res.",                 [("smear13", 1)],                   ["hi"] + EMB_VARIATIONS, 1),
        ("sqsmear15",           "dE/dx Stochastic Smearing",           [("sqsmear15", 1)],                 ["hi"] + EMB_VARIATIONS, 1),
        ("smear13_sqsmear15",   "Const. Res. + Stochastic",            [("smear13", 1), ("sqsmear15", 1)], ["hi"] + EMB_VARIATIONS, 1),
        ("smear13x2",           "2$\\times$ Const. Res.",              [("smear13", 2)],                   ["hi"] + EMB_VARIATIONS, 1),
        ("smear13x2_sqsmear15", "2$\\times$ Const. Res. + Stochastic", [("smear13", 2), ("sqsmear15", 1)], ["hi"] + EMB_VARIATIONS, 1),
        ("noemb_smear13",       "No EMB, dE/dx Constant Res.",         [("smear13", 1)],                   ["hi"], 1),
    ]

    # ICARUS-only: the dE/dx bias variation does not exist in the SBND
    # production, so these two models are simply not scanned for SBND.
    if has_dedxbias:
        models += [
            ("dedxbias",         "dE/dx Bias",               [("dedxbias", 1)],                 ["hi"] + EMB_VARIATIONS, 1),
            ("dedxbias_smear13", "dE/dx Bias + Const. Res.", [("dedxbias", 1), ("smear13", 1)], ["hi"] + EMB_VARIATIONS, 1),
        ]

    # COMBINED LARGE-UNCERTAINTY MODEL
    # The biggest dE/dx term each detector has, taken together with a doubled
    # GEANT4 budget. The dE/dx part is necessarily detector-dependent: ICARUS
    # combines the constant-resolution smearing with the dE/dx bias, while SBND
    # -- which has no dE/dx bias sample -- instead takes the constant resolution
    # at twice its size. Both carry the same filename tag so the two detectors
    # line up plot for plot.
    if has_dedxbias:
        combo_terms = [("smear13", 1), ("dedxbias", 1)]
        combo_label = "Const. Res. + dE/dx Bias, 2$\\times$ G4"
    else:
        combo_terms = [("smear13", 2)]
        combo_label = "2$\\times$ Const. Res., 2$\\times$ G4"
    models += [("combined_g4x2", combo_label, combo_terms,
                ["hi"] + EMB_VARIATIONS, G4_NORM_SCALE)]

    if has_rp25:
        # SBND-ONLY: EMB recombination R blown up by a factor of 10, together
        # with the constant-resolution smearing at its nominal size.
        # NB: R_p is dropped from this model's BASE calo list and re-entered as a
        # scaled smearing term. Leaving it in both would enter it into the
        # covariance twice -- once at nominal and once at 10x -- rather than
        # replacing it.
        models += [
            ("embR10_smear13", "10$\\times$ EMB $R$ + Const. Res.",
             [("smear13", 1), (EMB_R, EMB_R_NORM)],
             ["hi"] + [v for v in EMB_VARIATIONS if v != EMB_R],
             1),
            # sbn-rewgted-20 additions: the EMB R+0.25 variation (chi2R_p25
            # columns), scanned two ways. "R_p25" is the R+0.25 term by itself
            # (on top of the "hi" base every model carries); "R_p25_emb_smear"
            # takes it together with the full standard EMB budget and the
            # constant-resolution smearing. NB: in the combined model R+0.25
            # enters IN ADDITION to the nominal R_p (both in quadrature), not as
            # a replacement -- unlike embR10_smear13 above, which swaps R_p out
            # for its scaled copy.
            ("R_p25", "EMB $R$+0.25",
             [("R_p25", 1)],
             ["hi"],
             1),
            ("R_p25_emb_smear", "dE/dx Smearing + EMB + EMB $R$+0.25",
             [("smear13", 1), ("R_p25", 1)],
             ["hi"] + EMB_VARIATIONS,
             1),
        ]

    return models


# THE dE/dx SMEARING SCAN
# The detector's nominal budget with the dE/dx constant-resolution smearing term
# at 0x, 1x and 2x, and EVERY OTHER calorimetric term held exactly where the
# nominal budget puts it -- same base calo variations (gain, and EMB on SBND),
# same dE/dx bias on ICARUS, same G4 norm. Only the smearing moves, so the three
# bands differ by that term and nothing else.
#
# One of the three IS the detector's nominal budget: ICARUS takes the smearing at
# 1x, SBND at 2x. The tags are deliberately detector-independent so the two line
# up plot for plot.
DEDX_SMEAR_SCAN = [
    ("dedx_nosmear", "No dE/dx Smearing",         0),
    ("dedx_smear1x", "1$\\times$ dE/dx Smearing", 1),
    ("dedx_smear2x", "2$\\times$ dE/dx Smearing", 2),
]


def smear_scan_models(detector):
    """The nominal budget at three sizes of the dE/dx smearing term."""
    (_tag, _label, terms, calo, g4) = NOMINAL_CALO_MODEL[
        "SBND" if detector == "SBND" else "ICARUS"]
    # every nominal term EXCEPT the constant-resolution smearing, carried through
    # unchanged (ICARUS: the dE/dx bias; SBND: nothing)
    held = [(v, n) for (v, n) in terms if v != "smear13"]
    return [(tag, label, ([("smear13", norm)] if norm else []) + held, list(calo), g4)
            for (tag, label, norm) in DEDX_SMEAR_SCAN]


CALO_MODEL = os.environ.get("GUMPLE_CALO_MODEL", "scan")

# Which models are plotted where. "smearscan" is the only set that differs
# between the two: the angular quintile stages keep the single nominal budget
# (three bands per angular bin would be unreadable and the statistics per bin do
# not support the comparison), while the angle-inclusive stage gets the scan.
if CALO_MODEL == "smearscan":
    SMEAR_MODELS_ANGULAR = build_smear_models(DETECTOR, "nominal")
    SMEAR_MODELS_INCLUSIVE = smear_scan_models(DETECTOR)
else:
    SMEAR_MODELS_ANGULAR = build_smear_models(DETECTOR, CALO_MODEL)
    SMEAR_MODELS_INCLUSIVE = SMEAR_MODELS_ANGULAR

# every model a systematics object has to be built for, deduplicated by tag
SMEAR_MODELS = list(SMEAR_MODELS_ANGULAR)
_seen = set(m[0] for m in SMEAR_MODELS)
for _m in SMEAR_MODELS_INCLUSIVE:
    if _m[0] not in _seen:
        SMEAR_MODELS.append(_m)
        _seen.add(_m[0])

# Every calo/smear variation used anywhere below. Drives the nominal-fill of the
# dirt sample, the construction of the variation frames, and the load-time drop
# of the unused chi2 columns.
#
# NB THE ORDER MATTERS for "scan": it feeds CHI2_DROPS, and `drops` is part of
# the loaddf cache key (loaddf.py _cache_key). Reordering this list would
# invalidate every cached -calovarB split, so the scan list is written out
# literally, exactly as it has always been, rather than derived from the models.
# The assert below is what keeps the two in step.
if CALO_MODEL == "scan":
    chi2_variations = ["smear13", "sqsmear15", "hi", "alpha_p", "beta_p", "R_p"]
    if HAS_DEDXBIAS:
        chi2_variations = chi2_variations + ["dedxbias"]
    if HAS_RP25:
        chi2_variations = chi2_variations + ["R_p25"]
else:
    # derived: the base calo variations and smearing terms the models actually use
    chi2_variations = []
    for (_tag, _label, terms, calo, _g4) in SMEAR_MODELS:
        for v in list(calo) + [t for (t, _n) in terms]:
            if v not in chi2_variations:
                chi2_variations.append(v)

# a model may not reference a variation frame that is never built
_model_vars = set(v for (_t, _l, terms, calo, _g) in SMEAR_MODELS
                  for v in list(calo) + [t for (t, _n) in terms])
assert not (_model_vars - set(chi2_variations)), \
    "SMEAR_MODELS use variations that are not loaded: %s" % sorted(_model_vars - set(chi2_variations))
# and SBND must never take the dE/dx bias universe (it was not generated there)
assert HAS_DEDXBIAS or "dedxbias" not in chi2_variations, \
    "dedxbias is not a valid systematic universe for %s" % DETECTOR

print("DETECTOR:    %s" % DETECTOR)
print("CHI2 FLAVOR: %s -- nominal %s, variations %s"
      % (CHI2_FLAVOR, chi2_col("mu", "mu", CHI2_NOMVAR),
         ", ".join(chi2_variations)))
print("CALO MODEL:  %s -- %d model(s) (%d inclusive, %d angular), quintiles %s"
      % (CALO_MODEL, len(SMEAR_MODELS), len(SMEAR_MODELS_INCLUSIVE),
         len(SMEAR_MODELS_ANGULAR), "ON" if DO_ANGLE_BINS else "OFF"))

In [ ]:
PLOTBASE = os.environ.get(
    "GUMPLE_PLOTBASE", "/Users/gputnam/Work/osc/cafpyana/plots-gumple-2026-08-29-calovarB")

# One full set of plots per chi2 flavor, each in its own sub-directory.
PLOTDIR = PLOTBASE + "/pid/" + CHI2_FLAVOR + "/"

DOSAVE = True

# AREA (SHAPE) NORMALIZATION ONLY.
# These comparisons run on the full (un-prescaled) on-beam streams, so the
# absolute rate must not be shown: every histogram is normalized to unit area
# and the chi2 is the shape-only chi2. There is deliberately no absolute-norm
# path in this notebook -- the plotting loop asserts on this.
AREANORM = True

os.makedirs(PLOTDIR, exist_ok=True)
os.makedirs(PLOTDIR + "/png", exist_ok=True)
os.makedirs(PLOTDIR + "/pdf", exist_ok=True)
print("PLOTDIR:", PLOTDIR)


In [ ]:
import glob
import re

DF_DIR = os.environ.get("GUMPLE_DF_DIR", "/Users/gputnam/Work/osc/sbn-rewgted-20-calovarB/")
if not DF_DIR.endswith("/"):
    DF_DIR += "/"

# Cap on the number of CV MC files loaded, for when the full set does not fit in
# memory (0 / unset = all of them). See the load cell for why this exists.
NMC = int(os.environ.get("GUMPLE_NMC", "0")) or None


def mc_files(pattern):
    """The CV MC files matching `pattern` (a "..._%i.df" template), numerically ordered.

    Globbed rather than hardcoded as range(N): the split changes from production
    to production (calovarB: SBND 13 files, ICARUS Run2 3, Run4 4; -calovar had
    66/11/20), and a stale count silently throws away statistics."""
    fs = glob.glob(DF_DIR + pattern.replace("%i", "*"))
    fs = sorted(fs, key=lambda f: int(re.search(r"_(\d+)\.df$", f).group(1)))
    if not fs:
        raise RuntimeError("no CV MC files matching %s in %s" % (pattern, DF_DIR))
    return fs[:NMC]


if DETECTOR == "SBND":
    # Full-statistics on-beam stream (8.4e19 POT), not the FixedDev dev sample (4.6e18).
    ONBEAM = DF_DIR + "SBND_SpringBNBData_FullOnBeam.df"
    OFFBEAM_FILES = [DF_DIR + "SBND_SpringBNBOffData.df"]

    MC_FILES = mc_files("SBNDMCCV_%i.df")
    DIRT_FILES = [DF_DIR + "SBND_SpringLowEMC.df"]

    # Each entry: (detector-to-load-with, (CV file list, variation file list, ...))
    DETVAR_FILES = [
        ("SBND", ([DF_DIR + "SBND_SpringMC_Nom.df"],
                  [DF_DIR + "SBND_SpringMC_0xSCE.df"],
                  [DF_DIR + "SBND_SpringMC_2xSCE.df"])),
        ("SBND", ([DF_DIR + "SBND_SpringMC_Nom.df"],
                  [DF_DIR + "SBND_SpringMC_DENT.df"])),
        ("SBND", ([DF_DIR + "SBND_SpringMC_WMNom.df"],
                  [DF_DIR + "SBND_SpringMC_WMXThetaXW.df"])),
        ("SBND", ([DF_DIR + "SBND_SpringMC_WMNom.df"],
                  [DF_DIR + "SBND_SpringMC_WMYZ.df"])),
    ]
    DETVAR_NAMES = ["SCE", "DENT", "WM $X\\theta_{xw}$", "WM $YZ$"]

elif DETECTOR == "ICARUS Run2":
    # Full-statistics on-beam stream: the "unblind" file is a 1/10 prescale of this
    # one (identical gate_delta distribution, 1.985e19 vs 1.994e20 POT). The
    # *(1.-1/200.) gate correction below is a separate effect and still applies.
    ONBEAM = DF_DIR + "ICARUS_SpringRun2BNB_FullOnBeam.df"
    OFFBEAM_FILES = [DF_DIR + "ICARUS_SpringRun2BNBOff_unblind.df"]

    MC_FILES = mc_files("ICARUSRun2_SpringMCOverlay_rewgt_%i.df")
    DIRT_FILES = [DF_DIR + "ICARUSRun2_Spring_Overlay_Dirt.df"]

    DETVAR_FILES = [
        ("ICARUS Run2", (MC_FILES,  # no *_Overlay_Nom.df: pair against the rewgt CV
                         [DF_DIR + "ICARUSRun2_Spring_Overlay_SCE.df"])),
        ("ICARUS Run2", (MC_FILES,  # no *_Overlay_Nom.df: pair against the rewgt CV
                         [DF_DIR + "ICARUSRun2_Spring_Overlay_WMXThXW.df"])),
        ("ICARUS Run2", (MC_FILES,  # no *_Overlay_Nom.df: pair against the rewgt CV
                         [DF_DIR + "ICARUSRun2_Spring_Overlay_WMYZ.df"])),
    ]
    DETVAR_NAMES = ["SCE", "WM $X\\theta_{xw}$", "WM $YZ$"]

elif DETECTOR == "ICARUS Run4":
    # NB: no Run4 FullOnBeam file in this production, so Run 4 stays on the
    # prescaled "unblind" stream. This asymmetry with Run 2 is deliberate.
    ONBEAM = DF_DIR + "ICARUS_SpringRun4BNB_unblind.df"
    OFFBEAM_FILES = [DF_DIR + "ICARUS_SpringRun4BNBOff_unblind.df"]

    MC_FILES = mc_files("ICARUSRun4_SpringMCOverlay_rewgt_%i.df")
    DIRT_FILES = [DF_DIR + "ICARUSRun4_Spring_Overlay_Dirt.df"]

    DETVAR_FILES = [
        ("ICARUS Run4", (MC_FILES,  # no *_Overlay_Nom.df: pair against the rewgt CV
                         [DF_DIR + "ICARUSRun4_Spring_Overlay_SCE.df"])),
        # NB: real Run4 WireMod samples now exist -- this used to borrow the Run2
        # pair, which applied the Run2 MC flash-PE scale (0.632) instead of Run4's
        # (0.358) and so mis-set the flash cut on this sample.
        ("ICARUS Run4", (MC_FILES,  # no *_Overlay_Nom.df: pair against the rewgt CV
                         [DF_DIR + "ICARUSRun4_Spring_Overlay_WMXThXW.df"])),
        ("ICARUS Run4", (MC_FILES,  # no *_Overlay_Nom.df: pair against the rewgt CV
                         [DF_DIR + "ICARUSRun4_Spring_Overlay_WMYZ.df"])),
    ]
    DETVAR_NAMES = ["SCE", "WM $X\\theta_{xw}$", "WM $YZ$"]

print("DF_DIR:   %s" % DF_DIR)
print("MC_FILES: %d files (%.1f GB)"
      % (len(MC_FILES), sum(os.path.getsize(f) for f in MC_FILES)/1e9))


## POT / gate bookkeeping

In [ ]:
import h5py

def read_dfs(file, key):
    with h5py.File(file, "r") as f:
        keys = [k for k in f.keys() if k.startswith(key)]
        return pd.concat([pd.read_hdf(file, k) for k in keys])

In [ ]:
if DETECTOR == "ICARUS Run2":
    # NB: the old *(1-1/100.) on-beam prescale correction is dropped -- the new
    # "unblind" files are the full (non-prescaled) stream.
    ngates_ON = read_dfs(ONBEAM, "trig").gate_delta.sum()*(1.-1/200.)
    ngates_OFF = sum(read_dfs(f, "trig").gate_delta.sum() for f in OFFBEAM_FILES)*(1-1/20.)

    OFF_w = ngates_ON / ngates_OFF
elif DETECTOR == "ICARUS Run4":
    # NB: the old *(1-1/100.) on-beam prescale correction is dropped -- the new
    # "unblind" files are the full (non-prescaled) stream.
    ngates_ON = read_dfs(ONBEAM, "trig").gate_delta.sum()*(1.-1/40.)
    ngates_OFF = sum(read_dfs(f, "trig").gate_delta.sum() for f in OFFBEAM_FILES)*(1-1/20.)

    OFF_w = ngates_ON / ngates_OFF
elif "SBND" in DETECTOR:
    ngates_ON = read_dfs(ONBEAM, "bnb").shape[0]
    ngates_OFF = sum(read_dfs(f, "hdr").noffbeambnb.sum() for f in OFFBEAM_FILES)

    f_factor = 0.0754
    OFF_w = (1. - f_factor) * (ngates_ON) / (ngates_OFF)

ngates_ON, ngates_OFF, OFF_w

In [ ]:
if "ICARUS" in DETECTOR:
    print("ON:", 1/read_dfs(ONBEAM, "trig").gate_delta.mean(),
          "OFF:", [1/read_dfs(f, "trig").gate_delta.mean() for f in OFFBEAM_FILES])

In [ ]:
if "ICARUS" in DETECTOR:
    POT = read_dfs(ONBEAM, "hdr").pot.sum()
elif "SBND" in DETECTOR:
    POT = read_dfs(ONBEAM, "bnb").TOR875.sum()

POT

In [ ]:
if "SBND" in DETECTOR:  # TOR875 is SBND-only bookkeeping
    print(read_dfs(ONBEAM, "bnb").TOR875.sum()/ 1e19)

In [ ]:
print("N GATES ON / 5e12 POT")
print(5e12*ngates_ON/POT)

In [ ]:
NEVT_ON = read_dfs(ONBEAM, "hdr").shape[0]
NEVT_OFF = sum(read_dfs(f, "hdr").shape[0] for f in OFFBEAM_FILES)

NEVT = NEVT_ON - NEVT_OFF*OFF_w

In [ ]:
NEVT_ON, POT, NEVT_ON / (POT / 1e15)

In [ ]:
NEVT, POT, NEVT / (POT / 1e15)

## Load MC, detector variations, dirt, and data

In [ ]:
# NB: the flash cut is NOT applied in the load-time preselection so that the
# trigger (flash-PE scale) systematic can vary the threshold in both directions;
# it is applied at selection time (the per-stage cut columns) instead.
def FV(df):
    # gumple_cuts equivalent of the old slcfv & mufv & pfv & cathode chain:
    # cut_contained / cut_cathode are the production flags; slcfv_cut now also
    # excludes the ICARUS WW dangling-cable region.
    return gc.sanity_cut(df) & gc.slcfv_cut(df) & df.cut_contained & df.cut_cathode

In [ ]:
importlib.reload(loaddf)
importlib.reload(syst)
importlib.reload(gc)

In [ ]:
# LOAD-TIME SLIMMING OF THE CHI2 COLUMNS
# The calovarB CV MC carries 274 chi2 columns (4 planes x 17 variations x 4
# candidate combos) out of 430 columns total. This notebook only ever reads, per
# plane, the nominal plus the variations in chi2_variations; dropping the rest
# (144 columns) before the pd.concat at the end of loaddf.loadl cuts the CV frame
# by ~a third of its width.
#
# NB: the list is deliberately flavor-INDEPENDENT (it keeps every plane's kept
# variations, not just the selected flavor's), so all four flavor runs of a
# -calovarB detector share ONE loaddf cache -- `drops` is part of the cache key
# (loaddf.py:_cache_key), so a per-flavor list would force a full re-load and a
# separate multi-GB cache for every flavor. In a production that stores a single
# calculation (sbn-rewgted-21) there is nothing to share, and the p2trim* names
# below simply name columns that are not there: loaddf drops with
# errors='ignore', so they are harmless no-ops.
#
# NB: do NOT add loaddf.get_std_drops() here. It drops p_len, crthit and
# true_nu_E, which the cut stages (simple_cosrej_trkqual_p3, crtveto) below use.
_ALL_CHI2_VARIATIONS = ["cv", "lo", "hi", "2lo", "2hi", "smear5", "smear13", "sqsmear15",
                        "alpha_p", "alpha_m", "beta_p", "beta_m", "R_p", "R_m", "R_p25",
                        "dedxbias", "cafpyana"]
_CHI2_KEEP = set(chi2_variations) | {"", "cafpyana"}

CHI2_DROPS = ["%s_chi2%s%s_of_%s_cand" % (par, plane, var, cand)
              for (plane, _nom, _label) in CHI2_FLAVORS.values()
              for var in _ALL_CHI2_VARIATIONS if var not in _CHI2_KEEP
              for par in ("mu", "prot") for cand in ("mu", "prot")]

# calovarB went the other way from -calovar: far fewer, much larger CV files
# (SBND 13 x ~2.8 GB, ICARUS Run4 4 x ~4.8 GB, Run2 2 x ~3.5 GB + 1 x 0.6 GB,
# where -calovar had 66/20/11 pieces of ~0.5-1 GB). loadl concatenates every
# per-file frame at the end, so peak memory is ~2x the final frame; a smaller
# pool bounds how many of these large per-file frames are alive at once. If the
# full set still does not fit, cap it with GUMPLE_NMC rather than changing
# anything here.
MC_NJOB = 3

print("dropping %d unused chi2 columns at load time" % len(CHI2_DROPS))

df, match, mcpot = loaddf.loadl(MC_FILES, njob=min(len(MC_FILES), MC_NJOB), detector=DETECTOR,
                                preselection=FV, reweight_aFF=True, pot_univ=True,
                                drops=CHI2_DROPS)

# point the nominal chi2 columns at the selected flavor
apply_flavor(df)

In [ ]:
# LOAD DETECTOR VARIATION SAMPLES
# Each DETVAR_FILES entry is (detector-to-load-with, (CV file list, variation file list, ...));
# the loaded frames in each group are matched to their common events.

def load_detvar(detvar, detector):
    dfs, matches, pots = zip(*[loaddf.loadl(flist, preselection=FV, include_syst=False, detector=detector)
                               for flist in detvar])
    dfs, pots = loaddf.match_common_evts(matches, dfs, pots)
    for d in dfs:
        apply_flavor(d)
    return dfs, pots

if len(DETVAR_FILES) > 0:
    detvars, detvar_pots = zip(*[load_detvar(dv, dv_det) for (dv_det, dv) in tqdm(DETVAR_FILES)])
else:
    detvars, detvar_pots = [], []


In [ ]:
loaddf.scale_pot(df, mcpot, POT)
for i in range(len(detvars)):
    print(DETVAR_NAMES[i])
    for j in range(len(detvars[i])):
        loaddf.scale_pot(detvars[i][j], detvar_pots[i][j], POT)

In [ ]:
# ADD IN PID/CALORIMETRY VARIATIONS
def v_variation(df, setvars):
    df = df[[c for c in df.columns if "univ" not in c]].copy()
    for (new, old) in setvars:
        df[new] = df[old]
    return df

def v_calovariation(df, variation):
    """Copy of df with the nominal chi2 columns replaced by `variation`'s.

    Both sides follow the selected flavor: the destination is always the
    flavor-free nominal name (what the plotting and cut machinery reads), the
    source is the selected flavor's column for that variation, e.g.
    mu_chi2p2trim2_smear13_of_mu_cand for flavor "p2trim2". SMEAR_MODELS below
    therefore stays written in the plain variation names."""
    setvars = [("%s_chi2_of_%s_cand" % (par, cand), chi2_col(par, cand, variation))
               for par in ("mu", "prot") for cand in ("mu", "prot")]
    return v_variation(df, setvars)


In [ ]:
# SMEARING SYSTEMATIC MODELS -- built in the RUN CONFIGURATION cell above by
# build_smear_models(DETECTOR, CALO_MODEL), which is also where the models, the
# variation names and their sizes are documented. This cell only reports what
# came out, so the choice is in the log next to the yields it produced.
print("%d smearing model(s) for %s (GUMPLE_CALO_MODEL=%s):"
      % (len(SMEAR_MODELS), DETECTOR, CALO_MODEL))
_incl = set(m[0] for m in SMEAR_MODELS_INCLUSIVE)
_ang = set(m[0] for m in SMEAR_MODELS_ANGULAR)
for (tag, label, terms, calo, g4_norm) in SMEAR_MODELS:
    where = "+".join([w for w, s in (("inclusive", _incl), ("angular", _ang)) if tag in s])
    print("  %-14s base %-30s terms %-28s G4 x%g   [%s]"
          % (tag, "+".join(calo) or "-",
             "+".join("%s%s" % (v, "" if n == 1 else " x%g" % n) for (v, n) in terms) or "-",
             g4_norm, where))

In [ ]:
if INCLUDE_DIRT and len(DIRT_FILES) > 0:
    dirt, dirtmatch, dirtpot = loaddf.loadl(DIRT_FILES, njob=min(len(DIRT_FILES), 10),
                                            detector=DETECTOR, preselection=FV, include_syst=False)
    loaddf.scale_pot(dirt, dirtpot, POT)

    # point the nominal chi2 columns at the selected flavor -- BEFORE the
    # variation fill below, which copies from them
    apply_flavor(dirt)

    # dirt has no systematic-universe weights: set them to 1
    for c in df.columns:
        if "univ" in c:
            dirt[c] = 1

    # dirt has no chi2-variation columns: set them to nominal (no variation).
    # The names are the SELECTED FLAVOR's variation columns, since that is what
    # v_calovariation reads.
    for par in ("mu", "prot"):
        for cand in ("mu", "prot"):
            nom = dirt["%s_chi2_of_%s_cand" % (par, cand)].to_numpy()
            for chi2_var in chi2_variations:
                dirt[chi2_col(par, cand, chi2_var)] = nom

    dirt["dirt"] = True
    df["dirt"] = False
elif INCLUDE_DIRT:
    df["dirt"] = False


In [ ]:
print("Number of data triggers/1e15 POT: ", NEVT*(1e15/POT))
print("Number of scaled MC triggers/1e15 POT: ", match.shape[0]*(1e15/mcpot))

In [ ]:
print("Number of data triggers/1e15 POT: ", NEVT*(1e15/POT))

if INCLUDE_DIRT and len(DIRT_FILES) > 0:
    N_dirtevt = dirtmatch.shape[0]*(1e15/dirtpot)
else:
    N_dirtevt = 0

print("Number of scaled MC triggers/1e15 POT (w/dirt): ", N_dirtevt + match.shape[0]*(1e15/mcpot))

In [ ]:
print("Number of data triggers: ", NEVT)
print("Number of scaled MC triggers: ", match.shape[0]*(POT/mcpot))

In [ ]:
print("Number of data triggers: ", NEVT)

if INCLUDE_DIRT and len(DIRT_FILES) > 0:
    N_dirtevt = dirtmatch.shape[0]*(POT/dirtpot)
else:
    N_dirtevt = 0

print("Number of scaled MC triggers (w/dirt): ", match.shape[0]*(POT/mcpot) + N_dirtevt)

In [ ]:
if INCLUDE_DIRT and len(DIRT_FILES) > 0:
    # add dirt to the CV; it is also included (below) as a 100% syst. unc. via the OOAV sample
    df = pd.concat([df[~df.dirt], dirt])

In [ ]:
# Chi2/calorimetry variation frames built from the full CV df (incl. dirt,
# whose variation columns were set to nominal); SampleSystematic needs no
# separate CV. Built once and shared between the smearing models.
chi2_frames = {V: v_calovariation(df, V) for V in chi2_variations}


In [ ]:
def TrueAV(df):
    vtx = pd.DataFrame({
        "detector": df.detector,
        "Run": df.Run,
        "x": df.true_vtx_x,
        "y": df.true_vtx_y,
        "z": df.true_vtx_z,
    }, index=df.index)
    return gc._fv_cut(vtx, 0, 0, 0, 0)

def OOAV(df):
    return ~np.isnan(df.true_vtx_x) & ~TrueAV(df)

df_ooav = df[OOAV(df)].copy()

In [ ]:
# ============================================================
# TRIGGER SYSTEMATIC: flash-PE scale-factor uncertainty
# ============================================================
# Best-fit MC PE scale factors from the data/MC fits in FlashMCDataComparison.ipynb
# (applied to flash_maxpe at load time in loaddf.py):
#   SBND: 0.642 +/- 0.005   ICARUS Run2: 0.632 +/- 0.024   ICARUS Run4: 0.358 +/- 0.017
#
# Varying the scale s -> s*(1 -+ f), f = unc/s, is equivalent to varying the
# threshold on flash_maxpe: T -> T/(1 -+ f). The flash cut is applied in the
# per-stage cut columns (NOT the load-time preselection), so both the
# tightened- and loosened-threshold universes are available.
# NB: for SBND the threshold is -1 so the variation is a numerical no-op.
TRIG_PE_SCALE     = {1: 0.642, 2: 0.632, 4: 0.358}  # Run -> best-fit s (matches loaddf.py)
TRIG_PE_SCALE_UNC = {1: 0.005, 2: 0.024, 4: 0.017}  # Run -> unc. on s
TRIG_THRESHOLD    = {1: 2000., 2: 5000., 4: 1000.}  # Run -> gump_cuts.flash_cut threshold

def trig_flash_cut_var(df, updn, runs=None):
    """Flash cut with the threshold varied by the +/-1 sigma PE-scale uncertainty.

    updn = +1: scale DOWN 1 sigma -> threshold UP   (fewer events)
    updn = -1: scale UP   1 sigma -> threshold DOWN (more events)
    runs: restrict the variation to these runs (others stay at the CV threshold).
    """
    sel = pd.Series(False, index=df.index)
    for run, T in TRIG_THRESHOLD.items():
        f = TRIG_PE_SCALE_UNC[run] / TRIG_PE_SCALE[run]
        Tvar = T / (1 - updn*f) if (runs is None or run in runs) else T
        sel = sel | ((df.Run == run) & (df.flash_maxpe > Tvar))
    return sel

In [ ]:
ONdf, _, _ = loaddf.load(ONBEAM, load_truth=False, include_syst=False, detector=DETECTOR, preselection=FV)

_offs = [loaddf.load(f, load_truth=False, include_syst=False, detector=DETECTOR, preselection=FV, offbeampot=True)
         for f in OFFBEAM_FILES]
OFFdf = pd.concat([o[0] for o in _offs])
OFFPOT = sum(o[2] for o in _offs)  # cross-check only; OFF_w computed above is used for scaling

# point the nominal chi2 columns at the selected flavor
apply_flavor(ONdf)
apply_flavor(OFFdf)


## Plotting machinery

In [ ]:
pdg_list = [2212, 13, 211]
pdg_labels = ["$p$", "$\\mu$", "$\\pi^\\pm$", "Other"]
pdg_colors = ["#315031", "#d54c28", "#1e3f54", "#c89648"]

def breakdown_pdg(var, df, particle="p"):
    ret = [var[np.abs(df["%s_true_pdg" % particle] == i)] for i in pdg_list]
    ret.append(var[sum([np.abs(df["%s_true_pdg" % particle] == i) for i in pdg_list]) == 0])
    return ret

def breakdown_pdg_p(var, df):
    return breakdown_pdg(var, df, "p")

def breakdown_pdg_mu(var, df):
    return breakdown_pdg(var, df, "mu")

In [ ]:
FONTSIZE = 14
HAWKS_COLORS = ["#315031", "#d54c28", "#1e3f54", "#c89648", "#43140b", "#95af8b"]

def add_style(ax, xlabel, title="", det="ICARUS"):
    ax.tick_params(axis='both', which='both', direction='in', length=6, width=1.5, labelsize=FONTSIZE, top=True, right=True)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
    ax.set_xlabel(xlabel, fontsize=FONTSIZE, fontweight='bold')
    ax.set_ylabel('Area Normalized', fontsize=FONTSIZE, fontweight='bold')
    ax.set_title(f"$\\bf{{{det}}}$  {title}", fontsize=FONTSIZE+2)
    ax.legend(fontsize=FONTSIZE)


In [ ]:
def f_chi2(NMC, Ndata, cov):
    # ignore singular entries
    which_bin = NMC > 0

    NMC = NMC[which_bin]
    Ndata = Ndata[which_bin]
    cov = cov[which_bin, :]
    cov = cov[:, which_bin]

    delta = NMC - Ndata
    try:
        cov_inv = np.linalg.inv(cov)
    except np.linalg.LinAlgError as _:
        return -1, which_bin.sum()
        
    return delta@cov_inv@delta, which_bin.sum()

In [ ]:
def make_plot_data(var, bins, cut, mc_weight, breakdown, areanorm, breakdown_labels, breakdown_colors, xlabel, title, 
                   det="ICARUS", fillna=np.nan, syst=None):
    if syst is None:
        raise ValueError("pass the smearing model's systematics via syst=model_systs[tag]")
    
    pvars = breakdown(df.loc[df[cut], var].fillna(fillna), df[df[cut]])
    weights = breakdown(df.loc[df[cut], mc_weight], df[df[cut]])

    NMC_breakdown = []
    for pvar, w in zip(pvars, weights):    
        thisNMC, bins = np.histogram(pvar, bins=bins, weights=w)
        NMC_breakdown.append(thisNMC)
        
    NMC,_ = np.histogram(df.loc[df[cut], var].fillna(fillna), bins=bins, weights=df.loc[df[cut], mc_weight])
    NMC_abs = NMC
    if areanorm:
        diff = (bins[1:] - bins[:-1])
        norm = np.sum(NMC*diff)
        if norm > 1e-5:
            NMC = NMC / norm
            for i in range(len(NMC_breakdown)):
                NMC_breakdown[i] = NMC_breakdown[i] / norm

    NMC_breakdown = np.array(NMC_breakdown)
        
    NON,_ = np.histogram(ONdf.loc[ONdf[cut], var].fillna(fillna), bins=bins)
    NOff,_ = np.histogram(OFFdf.loc[OFFdf[cut], var].fillna(fillna), bins=bins)

    N = NON - NOff*OFF_w
    Nerr = np.sqrt(NON + NOff*OFF_w**2)
    if areanorm:
        diff = (bins[1:] - bins[:-1])
        
        norm = np.sum(N*diff)
        if norm > 1e-5:
            N = N / norm
            Nerr = Nerr / norm

    cov = syst.cov(var, cut, bins, NMC_abs, shapeonly=areanorm, fillna=fillna)
    err = np.sqrt(np.diag(cov))

    cov_w_stat = cov + np.diag(Nerr**2) # add stat uncertainty
    chi2, ndof = f_chi2(NMC, N, cov_w_stat)

    return {
        "det": det,
        "title": title,
        "xlabel": xlabel,
        "bins": bins,
        "areanorm": areanorm,
        "breakdown_labels": breakdown_labels,
        "breakdown_colors": breakdown_colors,
        "NMC_breakdown": NMC_breakdown,
        "NMC_total": NMC,
        "NData": N,
        "NDataErr": Nerr,
        "cov": cov,
        "cov_w_stat": cov_w_stat,
        "chi2": chi2,
        "ndof": ndof,
        "POT": POT
    }


In [ ]:
# Upper-left annotation stack: first line at ANNOT_Y0, each further line ANNOT_DY
# below it, all drawn with verticalalignment="top" at FONTSIZE-2. 0.09 is the
# smallest spacing that does not let 12 pt lines touch in an axes this tall.
ANNOT_Y0 = 0.82
ANNOT_DY = 0.09


def ratio_plot(plt, plotdata):
    fig, (ax0, ax1) = plt.subplots(2, 1, height_ratios=[3, 1], sharex=True)
    bins = plotdata["bins"]
    centers = (bins[:-1] + bins[1:])/2

    NMC_breakdown = plotdata["NMC_breakdown"]
    fill = np.array([centers for _ in range(NMC_breakdown.shape[0])]).T
    ax0.hist(fill, bins=bins, stacked=True, label=plotdata["breakdown_labels"],
                    color=plotdata["breakdown_colors"], weights=NMC_breakdown.T)

    NData = plotdata["NData"]
    NDataErr = plotdata["NDataErr"]
    line = ax0.errorbar(centers, NData, NDataErr, color="black", linestyle="none", marker=".")

    NMC = plotdata["NMC_total"]
    err = np.sqrt(np.diag(plotdata["cov"]))
    # full edge array with the last value repeated -- step="post" holds each y from x[i] to x[i+1], so stepping over the left edges alone leaves the last bin unshaded
    _hi, _lo = NMC+err, NMC-err
    ax0.fill_between(bins, np.append(_hi, _hi[-1]), np.append(_lo, _lo[-1]), facecolor="none", hatch="//", edgecolor="gray", linewidth=0.0, step="post")

    if "Maximum" in plotdata["xlabel"]:
        ax0.set_yscale("log")

    ax1.errorbar(centers, NData/NMC, NDataErr/NMC, color="black", linestyle="none", marker=".")
    ax1.set_ylim([0.5, 1.5])
    ax1.axhline([1], color="red", linestyle="--")
    # full edge array with the last value repeated -- step="post" holds each y from x[i] to x[i+1], so stepping over the left edges alone leaves the last bin unshaded
    _hi, _lo = 1+err/NMC, 1-err/NMC
    ax1.fill_between(bins, np.append(_hi, _hi[-1]), np.append(_lo, _lo[-1]), facecolor="none", hatch="//", edgecolor="gray", linewidth=0.0, step="post")

    ax0.tick_params(axis='both', which='both', direction='in', length=6, width=1.5, labelsize=FONTSIZE, top=True, right=True)
    ax1.tick_params(axis='both', which='both', direction='in', length=6, width=1.5, labelsize=FONTSIZE, top=True, right=True)
    for spine in ax0.spines.values():
        spine.set_linewidth(1.5)
    ax1.set_xlabel(plotdata["xlabel"], fontsize=FONTSIZE, fontweight='bold')
    
    if plotdata["areanorm"]:
        ax0.set_ylabel('Area Normalized', fontsize=FONTSIZE, fontweight='bold')
    else:
        ax0.set_ylabel('Events / %.1f$\\times 10^{19}$ POT' % (plotdata["POT"]/1e19), fontsize=FONTSIZE, fontweight='bold')

    det = plotdata["det"]
    title = plotdata["title"]
    ax0.set_title(f"$\\bf{{{det}}}$ {title}", fontsize=FONTSIZE+2)
    ld = ax0.legend([line], ["Data\n(ON Beam - OFF)"], frameon=False, loc="upper left", fontsize=10)

    # Headroom for the upper-left annotation stack. There are four lines now (chi2,
    # chi2 flavor, smearing model, angular bin), spaced 0.09 apart from ANNOT_Y0 --
    # they run down to ~0.50 in axes fraction, so the tallest bin has to stay below
    # that. 1/2.1 = 0.476. (Was 1.7 when the stack was three lines.)
    ax0_l0, ax0_hi = ax0.get_ylim()
    ax0.set_ylim([ax0_l0, ax0_hi*2.1])
    
    ax0.legend(fontsize=12, loc="upper right", ncol=2, reverse=True)
    ax0.add_artist(ld)

    chi2_str = "$\\chi^2_\\mathrm{shape}$" if plotdata["areanorm"] else "$\\chi^2$"
    ax0.text(0.05, ANNOT_Y0, "%s: %.1f / %i" % (chi2_str, plotdata["chi2"], plotdata["ndof"] - int(plotdata["areanorm"])),
            verticalalignment="top", horizontalalignment="left", fontsize=FONTSIZE-2, transform=ax0.transAxes)
    
    plt.subplots_adjust(hspace=0.05)
    return fig, ax0, ax1

## Cut stage, angular binning, and per-model systematics


In [ ]:
def simple_cosmic_rej(df):
    # n_pfp == 2: exactly one muon and one proton candidate (the GUMP 1p
    # multiplicity), applied at the base of every PID cut stage. NB kept out
    # of FV() -- that callable feeds the loaddf cache key.
    return FV(df) & (df.nu_score > 0.6) & (df.n_pfp == 2)

# Simple cosmic rejection plus track-quality requirements: a shower-like or very
# short prong has unreliable calorimetry, so its chi2 is not a meaningful PID
# variable. gump_cuts.pid_cut_df applies the muon cuts at 0.5 and 40 cm; these
# stages are the tighter 0.6 and 50 cm, with and without a short-proton veto.
MU_TRACK_SCORE_MIN = 0.6
MU_LEN_MIN = 50.       # cm
P_LEN_MIN_LOOSE = 3.   # cm -- short-proton veto of the one stage that applies one


def simple_cosrej_trkqual_nop(df):
    """Muon-candidate quality only -- NO cut on the proton candidate.

    Track-like and long enough for reliable muon calorimetry, with the proton
    candidate left completely unconstrained, so the proton-candidate chi2
    distributions keep their short/shower-like prongs. This is the stage that
    shows what the short-proton veto is actually removing."""
    return simple_cosmic_rej(df) \
        & (df.mu_trackScore > MU_TRACK_SCORE_MIN) \
        & (df.mu_len > MU_LEN_MIN)


def simple_cosrej_trkqual_p3(df):
    """Muon quality (as above) plus a 3 cm short-proton veto."""
    return simple_cosrej_trkqual_nop(df) & (df.p_len > P_LEN_MIN_LOOSE)

def crtveto(df):
    return FV(df) & ~df.crthit

def twoprong_cut(df):
    # GUMPLE: cut_0shwother replaces the dropped other_shw/trk_length columns
    return crtveto(df) & df.cut_0shwother

def pid_cut(df):
    return twoprong_cut(df) & gc.pid_cut(df)

In [ ]:
# CUT STAGES FOR THE PID COMPARISONS
# Every comparison below is repeated at each stage. The three stages are nested,
# each adding requirements to the one above it, so comparing them isolates what
# each group of cuts does to the chi2 distributions:
#
#   1. simplecosrej            FV & nu_score > 0.6 & n_pfp == 2   [& flash]
#   2. ..._trkqual_nop         + mu_trackScore > 0.6 & mu_len > 50 cm
#   3. ..._trkqual_p3          + p_len > 3 cm
#
# where the FV term (the FV() callable above, also the load-time preselection) is
#   gc.sanity_cut     slc_vtx_{x,y,z} all non-nan
#   gc.slcfv_cut      slice vertex in the FV: 10 cm in from x/y and the front of
#                     z, 50 cm from the back of z, per-detector/run boundaries,
#                     minus the ICARUS WW dangling-cable region
#   cut_contained     production flag: every pfp starts and ends in the same TPC
#                     as the vertex and is contained by 10 cm
#   cut_cathode       production flag: no pfp crosses the cathode (SBND only;
#                     True by construction for ICARUS)
# and the flash cut (gc.flash_cut: flash_maxpe > 5000 PE ICARUS Run2, > 1000 Run4,
# > 2000 SBND) is AND-ed onto every stage in the set_stages loop below -- it is
# kept out of FV() only so the trigger systematic can move the threshold.
#
# NB the 5 cm proton working point (the old "p5" stage) is dropped; stage 2
# replaces it with no proton cut at all, so the pair (2, 3) brackets the veto
# rather than comparing two nearby values of it.
#
# Entries: (cut function, dataframe column name, title drawn on the plot,
#           filename tag). The column name must be UNIQUE per stage -- it is the
# boolean column the systematics select on.
ALL_CUT_STAGES = [
    (simple_cosmic_rej,         "Simple Cos. Rej.",
                                "Simple Cos. Rej.",                  "simplecosrej"),
    (simple_cosrej_trkqual_nop, "Simple Cos. Rej. + Trk. Qual. nop",
                                "Simple Cos. Rej. + Muon Trk. Qual.", "simplecosrej_trkqual_nop"),
    (simple_cosrej_trkqual_p3,  "Simple Cos. Rej. + Trk. Qual. p3",
                                "Simple Cos. Rej. + Trk. Qual.",      "simplecosrej_trkqual_p3"),
]

# GUMPLE_CUT_STAGES selects which of them this run plots, by filename tag
# (comma-separated), or "all" for every stage. The stages are nested, so a subset
# is just fewer plots -- nothing downstream depends on their being all three.
# Everything below (the quintile edges, the stage columns, the systematics, the
# plotting loop) is driven off CUT_STAGES, so dropping a stage here drops its
# work as well as its output.
_stage_sel = os.environ.get("GUMPLE_CUT_STAGES", "all")
if _stage_sel == "all":
    CUT_STAGES = ALL_CUT_STAGES
else:
    _want = [t.strip() for t in _stage_sel.split(",") if t.strip()]
    _known = [tag for (_c, _b, _t, tag) in ALL_CUT_STAGES]
    _bad = [t for t in _want if t not in _known]
    if _bad:
        raise ValueError("unknown GUMPLE_CUT_STAGES %s (choose from %s)" % (_bad, _known))
    CUT_STAGES = [s for s in ALL_CUT_STAGES if s[3] in _want]

print("Cut stages: %s" % ", ".join(tag for (_c, _b, _t, tag) in CUT_STAGES))

# ============================================================
# ANGULAR BINNING
# ============================================================
# The chi2 PID comparisons are made inclusively AND -- when DO_ANGLE_BINS is on
# (GUMPLE_ANGLE_BINS, see the RUN CONFIGURATION cell) -- in five equal-statistics
# bins of two track angles, so an angle-dependent data/MC disagreement (wire
# response, recombination vs. angle) can be told apart from a flat mismodelling
# of the dE/dx resolution. With DO_ANGLE_BINS off, every stage list below
# collapses to the inclusive stage alone and no quintile edges are computed.
#
#   thetadrift = acos(|dir_x|)      -- the full 3D polar angle to the drift axis:
#                                      0 deg is along the drift, 90 deg is in the
#                                      wire plane. This is the angle the amount of
#                                      charge-per-wire actually depends on.
#   thetaxw    = |atan(dir_x/dir_z)| -- angle in the drift(x)-beam(z) plane. z is the
#                                      wire-pitch direction of the collection plane in
#                                      both detectors, i.e. the WireMod theta_xw.
#
# Both live in [0, 90] deg. thetadrift is there by construction (the |dir_x| folds
# forward- and backward-going together); thetaxw is folded into the first quadrant
# for the same reason -- the calorimetry responds to how steeply a track runs
# relative to the drift and pitch directions, not to which of the four sign
# combinations it does it in, and folding quadruples the statistics per bin.
#
# Each chi2 variable is binned in the angle of ITS OWN track: *_of_mu_cand in the
# muon candidate's angle, *_of_prot_cand in the proton candidate's. Hence two sets
# of quintile edges per angle.
# (column suffix, latex for the symbol with a "%s" slot for the candidate --
#  theta_xw already carries a subscript, so the candidate goes up top)
ANGLES = [
    ("thetadrift", "\\theta^{%s}_{drift}"),
    ("thetaxw",    "|\\theta^{%s}_{xw}|"),
]
CANDS = [("mu", "\\mu"), ("p", "p")]   # (df column prefix, latex for the candidate)
NQUANT = 5                             # quintiles


def angle_value(frame, cand, angle):
    """The track angle in degrees, in [0, 90].

    thetadrift = acos(|dir_x|), the polar angle to the drift axis. dir_* is a unit
    vector, but clip anyway so a component that rounds to 1+1e-16 cannot produce a
    nan.

    thetaxw = |atan(dir_x/dir_z)|, written as atan2 of the two magnitudes:
    identical to abs(atan(ratio)) but without the division, so a track lying
    exactly along z gives 90 rather than a divide-by-zero warning."""
    if angle == "thetadrift":
        return np.arccos(np.clip(np.abs(frame[cand + "_dir_x"]), 0., 1.))*180/np.pi
    if angle == "thetaxw":
        return np.arctan2(np.abs(frame[cand + "_dir_x"]),
                          np.abs(frame[cand + "_dir_z"]))*180/np.pi
    raise ValueError("unknown angle: %s" % angle)


def add_angles(frame):
    """Set mu/p_thetadrift and mu/p_thetaxw in place.

    Every frame that gets histogrammed (CV, OOAV, detector variations, calo
    variations, split tracks, on/off-beam data) needs these before its stage
    columns can be built."""
    for (cand, _) in CANDS:
        for (angle, _) in ANGLES:
            frame["%s_%s" % (cand, angle)] = angle_value(frame, cand, angle)
    return frame


def weighted_quantile(x, w, q):
    """Quantiles of x under weights w.

    Mirrors _weighted_quantile in eres_ar23_ar25.py -- importing that module
    would pull in a whole analysis script for six lines."""
    x = np.asarray(x)
    w = np.asarray(w)
    order = np.argsort(x)
    x, w = x[order], w[order]
    c = np.cumsum(w)
    return np.interp(np.asarray(q)*c[-1], c, x)


def stage_name(base, cand, angle, i):
    """Column name of one angular stage, e.g. "Simple Cos. Rej. muphi2"."""
    return "%s %s%s%d" % (base, cand, angle, i)


def angle_cut_label(cand_tex, angle_tex, lo, hi):
    """The cut this bin represents, as drawn on the plot."""
    v = angle_tex % cand_tex
    if not np.isfinite(lo):
        return "$%s < %.1f^\\circ$" % (v, hi)
    if not np.isfinite(hi):
        return "$%s > %.1f^\\circ$" % (v, lo)
    return "$%.1f^\\circ < %s < %.1f^\\circ$" % (lo, v, hi)


def var_cand(v):
    """Which candidate's angle a chi2 variable is binned in."""
    return "p" if "prot_cand" in v else "mu"


def stages_for(base, v):
    """Base stage plus the ten angular stages of `base` relevant to variable v.

    Just the base stage when the angular binning is off."""
    if not DO_ANGLE_BINS:
        return [base]
    cand = var_cand(v)
    return [base] + [stage_name(base, cand, angle, i)
                     for (angle, _) in ANGLES for i in range(NQUANT)]


def angular_stage_names(base):
    """Every angular stage column of `base` (empty when the binning is off)."""
    if not DO_ANGLE_BINS:
        return []
    return [stage_name(base, cand, angle, i)
            for (cand, _) in CANDS for (angle, _) in ANGLES for i in range(NQUANT)]


def set_stages(frame, base, base_sel, suffix=""):
    """Write the inclusive stage column plus the 2 x 2 x NQUANT angular ones.

    An angular stage is just the base selection AND-ed with a quintile of the
    candidate's angle. The systematics address the selection purely by column
    name (syst.histflat), so every existing systematic term works on an angular
    stage unmodified. Reads EDGES, set in the cell below. Assignment is by numpy
    array (positional) because the CV frames carry duplicate index labels.

    With DO_ANGLE_BINS off this writes the inclusive column only -- EDGES is
    empty in that case, so the loop below would have nothing to look up."""
    sel = np.asarray(base_sel, dtype=bool)
    frame[base + suffix] = sel
    if not DO_ANGLE_BINS:
        return frame
    for (cand, _) in CANDS:
        for (angle, _) in ANGLES:
            a = frame["%s_%s" % (cand, angle)].to_numpy()
            e = EDGES[(base, cand, angle)]
            for i in range(NQUANT):
                frame[stage_name(base, cand, angle, i) + suffix] = \
                    sel & (a >= e[i]) & (a < e[i+1])
    return frame

In [ ]:
# Angle columns on every frame that gets histogrammed. chi2_frames are built by
# v_calovariation, which drops only the "univ" columns, so the direction columns
# they need are still there.
angle_frames = [df, df_ooav, ONdf, OFFdf] \
    + [d for dv in detvars for d in dv] \
    + list(chi2_frames.values())

for frame in angle_frames:
    add_angles(frame)

# ------------------------------------------------------------------
# Quintile edges: POT-weighted MC CV, recomputed for EACH cut stage so that every
# plot's five bins hold 20% of that stage's MC. The two stages therefore have
# different edges -- each plot states its own cut.
# Outer edges are open so the five bins partition the data as well as the MC.
# ------------------------------------------------------------------
_mc_w = df.glob_scale.to_numpy()

EDGES = {}
if DO_ANGLE_BINS:
    for (cutf, base, _title, _tag) in CUT_STAGES:
        _mc_sel = np.asarray(cutf(df) & gc.flash_cut(df), dtype=bool)
        print("Angular quintile edges (POT-weighted MC CV at %s):" % base)
        for (cand, _) in CANDS:
            for (angle, _) in ANGLES:
                a = df["%s_%s" % (cand, angle)].to_numpy()
                good = _mc_sel & np.isfinite(a)
                EDGES[(base, cand, angle)] = np.concatenate([
                    [-np.inf],
                    weighted_quantile(a[good], _mc_w[good], np.arange(1, NQUANT)/NQUANT),
                    [np.inf]])
                print("  %-3s %-8s  %s   [%d MC rows used, %d with no angle]"
                      % (cand, angle,
                         ", ".join("%8.2f" % x for x in EDGES[(base, cand, angle)][1:-1]),
                         int(good.sum()), int((_mc_sel & ~np.isfinite(a)).sum())))
else:
    print("Angular quintile binning OFF (GUMPLE_ANGLE_BINS=0): "
          "inclusive stages only, no quintile edges computed")

ALL_STAGES = [s for (cutf, base, _title, _tag) in CUT_STAGES
              for s in [base] + angular_stage_names(base)]

# per-stage metadata: the plot title, the annotation, and the filename pieces
STAGE_INFO = {}
for (cutf, base, title, tag) in CUT_STAGES:
    STAGE_INFO[base] = {"base": base, "title": title, "cuttag": tag, "label": None, "tag": ""}
    if not DO_ANGLE_BINS:
        continue
    for (cand, cand_tex) in CANDS:
        for (angle, angle_tex) in ANGLES:
            e = EDGES[(base, cand, angle)]
            for i in range(NQUANT):
                STAGE_INFO[stage_name(base, cand, angle, i)] = {
                    "base": base,
                    "title": title,
                    "cuttag": tag,
                    "cand": cand,
                    "angle": angle,
                    "ibin": i,
                    "tag": "_%s%s%d" % (cand, angle, i),
                    "label": angle_cut_label(cand_tex, angle_tex, e[i], e[i+1]),
                }


for (cutf, base, _title, _tag) in CUT_STAGES:
    # nominal stages: base cut AND the nominal flash cut
    for frame in angle_frames:
        set_stages(frame, base, cutf(frame) & gc.flash_cut(frame))

    # trigger-systematic universes (MC CV only; the SelectionSystematic reads these)
    set_stages(df, base, cutf(df) & trig_flash_cut_var(df, +1), suffix="_trig_up")
    set_stages(df, base, cutf(df) & trig_flash_cut_var(df, -1), suffix="_trig_dn")

# WHAT IS IN EACH CUT STAGE
# Spelled out in the log rather than only in the source: the three stages are
# nested, so each line is the one above it plus the terms named. Every stage also
# carries the nominal flash-PE cut, AND-ed in by the set_stages loop above.
print("\nCut stage definitions (ALL stages; this run plots %s):"
      % ", ".join(tag for (_c, _b, _t, tag) in CUT_STAGES))
print("  %-28s %s" % ("simplecosrej",
      "FV & nu_score > 0.6 & n_pfp == 2"))
print("  %-28s %s" % ("", "  FV = sanity & slcfv & cut_contained & cut_cathode"))
print("  %-28s %s" % ("simplecosrej_trkqual_nop",
      "+ mu_trackScore > %.2f & mu_len > %.0f cm  (NO proton cut)"
      % (MU_TRACK_SCORE_MIN, MU_LEN_MIN)))
print("  %-28s %s" % ("simplecosrej_trkqual_p3",
      "+ p_len > %.0f cm" % P_LEN_MIN_LOOSE))
print("  %-28s %s" % ("[every stage]",
      "& gc.flash_cut (flash_maxpe > 5000/1000/2000 PE, IC Run2/Run4/SBND)"))

# Per-stage MC yields with and without the flash cut, so the size of each step is
# visible too.
print("\nNominal flash-PE cut (gc.flash_cut) on each base stage:")
_flash = np.asarray(gc.flash_cut(df), dtype=bool)
_prev_w = None
for (cutf, base, _title, _tag) in CUT_STAGES:
    _pre = np.asarray(cutf(df), dtype=bool)
    _pre_w = _mc_w[_pre].sum()
    _post_w = _mc_w[_pre & _flash].sum()
    _step = "" if _prev_w is None else "   [%.1f%% of the stage above]" % (100*_post_w/_prev_w)
    print("  %-36s MC %10.1f -> %10.1f  (%.1f%% kept)%s"
          % (base, _pre_w, _post_w, 100*_post_w/_pre_w, _step))
    _prev_w = _post_w

print("\nPer-stage yields:")
for stage in ALL_STAGES:
    print("  %-48s MC %12.1f   ON %8d   OFF %8d"
          % (stage, df.loc[df[stage], "glob_scale"].sum(),
             int(ONdf[stage].sum()), int(OFFdf[stage].sum())))

In [ ]:
# ============================================================
# BINDING-ENERGY SYSTEMATIC
# ============================================================
# +25 MeV shift of the binding energy in the reco neutrino-energy formula,
# applied to half the events (fraction=0.5), as in the signal box
# (SignalBoxSystematics-ReCAF.ipynb) and the event-selection notebook.
#
# NB: this term is numerically NULL for the plots below. shift_binding_energy
# recomputes nu_E_calo / nu_E_ccqe / del_p only; the plotted chi2 columns and
# the stage columns ride along as plain CV copies, so the universe and
# the CV give identical histograms. It is kept so the budget matches the signal
# box term-for-term rather than diverging again.
BE_SHIFT = 0.025
# NB: syst.recompute_kinematics is migrated to the GUMPLE psum convention --
# it requires the stored psum_E / psum_dir_* (and n_pfp) and raises without them.
BE_RECOMPUTE_COLS = ["glob_scale", "genie_mode", "mu_E", "p_E",
                     "mu_dir_x", "mu_dir_y", "mu_dir_z",
                     "p_dir_x", "p_dir_y", "p_dir_z",
                     "psum_E", "psum_p",
                     "psum_dir_x", "psum_dir_y", "psum_dir_z", "n_pfp",
                     # stored kinematics: the unshifted rows of the in-place
                     # BE universe keep these CV values verbatim (nu_E_ccqe is
                     # NOT stored -- rebuilt at nominal BE for unshifted rows)
                     "nu_E_calo", "del_p", "del_Tp", "del_phi"]

# every stage column (inclusive + angular) has to ride along into the BE universe
be_cols = sorted(set(BE_RECOMPUTE_COLS + chi2vars + ALL_STAGES))
be_df = syst.shift_binding_energy(df[be_cols], BE_SHIFT, fraction=0.5)

print("Binding-energy universe: %d rows (dE_miss = %.0f MeV on 50%% of events)"
      % (len(be_df), BE_SHIFT*1000))


# ============================================================
# TRACK-SPLITTING SYSTEMATIC (ICARUS)
# ============================================================
# Data shows an excess of muon-track endpoints at Z=0 and the cathodes that MC
# does not reproduce: muons crossing those planes are split by reconstruction
# more often in data. Split fractions measured by TrackSplittingCorrection.py
# (see TrackSplittingCorrection.md); the Run2+Run4 combined value per plane is
# applied as a one-sided, symmetrized 1-sigma variation, uncorrelated between
# planes. The plane definitions come from loaddf.SPLIT_REGIONS.
#
# This one is NOT null here: truncating the muon moves mu_end_*, so FV (and
# hence every stage column) has to be re-evaluated and the selected set changes.
#
# NB: the east cathode lies outside the Run 2 muon fiducial volume, so it has no
# Run 2 crossers and contributes nothing to the Run 2 covariance.
# TrackSplittingCorrection-2026-08-21.md (reCAF -14 measurement; rerun
# TrackSplittingCorrection_GUMPLE.py on -19 to update)
TRACKSPLIT_FRAC = {"Z=0": 0.1143, "East Cathode": 0.0446, "West Cathode": 0.0314}

tracksplit_systs = []
if "ICARUS" in DETECTOR:
    run = 2 if DETECTOR == "ICARUS Run2" else 4
    for name, (dim, coord) in loaddf.SPLIT_REGIONS.items():
        sdf, crosses = syst.split_tracks(df, dim, coord, runs=[run])
        if not crosses.any():
            # No crossers -> a null systematic. Skipped rather than added as a
            # zero term because the cuts do not survive an empty frame
            # (gc.cathode_cut reads df.detector.iloc[0]).
            print("Track split %-14s: no crossing muons in Run %d, skipped" % (name, run))
            continue
        # mu_end_* moved, so the stage cut must be re-evaluated on the split rows
        # (FV includes gc.mufv_cut, and TrackSplittingSystematic.univ indexes
        # splitdf by the stage being plotted)
        # mu_dir_* is untouched by the truncation, so the angle columns copied
        # over from df are still correct and only the stage columns are rebuilt
        for (cutf, base, _title, _tag) in CUT_STAGES:
            set_stages(sdf, base, cutf(sdf) & gc.flash_cut(sdf))
        tracksplit_systs.append(
            syst.TrackSplittingSystematic(df, sdf, crosses, TRACKSPLIT_FRAC[name]))
        print("Track split %-14s: f = %5.2f%%, %6d crossing muons "
              "(%5d still pass the cut after splitting)"
              % (name, 100*TRACKSPLIT_FRAC[name], int(crosses.sum()),
                 int(sdf[CUT_STAGES[0][1]].sum())))


# ============================================================
# COSMIC NORMALIZATION
# ============================================================
# From the off-beam / intime cosmic MC comparison (OffBeamCosmicMCComparison.ipynb).
# SBND only: the ICARUS MC is an overlay and already carries data cosmics, so it
# takes no separate cosmic normalization. (This replaces the unconditional 20%
# placeholder that used to be applied to both detectors.)
SBND_COSMIC_NORM = 0.107

cosmic_systs = []
if DETECTOR == "SBND":
    cosmic_systs.append(syst.SystSampleSystematic(df[df.true_iscosmic], norm=SBND_COSMIC_NORM))


In [ ]:
# Define systematic uncertainties, one budget per smearing model.
# NB: df must not be re-bound after this point -- these objects hold references
# to it (in-place column edits are fine).
# NB: no beam-off stat systematic here -- the off-beam statistical uncertainty
# is already propagated into the data error bars in make_plot_data.
class ScaledSystematic(syst.Systematic):
    """Wrap a systematic and scale its 1-sigma size by `norm` (covariance by norm^2).

    SystematicList only ever calls .cov() on its members, so overriding that is
    enough. Same convention SampleSystematic already uses for its own `norm`;
    G4Systematic itself takes no norm argument, hence the wrapper."""
    def __init__(self, systematic, norm=1):
        self.systematic = systematic
        self.norm = norm

    def cov(self, var, cut, bins, NCV, shapeonly=False, fillna=np.nan):
        return self.systematic.cov(var, cut, bins, NCV, shapeonly=shapeonly,
                                   fillna=fillna) * self.norm**2


def model_systematics(smear_terms, calo_variations, g4_norm=1):
    return syst.SystematicList([
        loaddf.FluxSystematic(df),
        ScaledSystematic(loaddf.G4Systematic(df), g4_norm),   # g4_norm=1 is a no-op
        # XSec: the weight-based knobs plus the binding-energy shift
        syst.SystematicList([loaddf.XSecSystematic(df), syst.SampleSystematic(be_df)]),
        syst.NormalizationSystematic(0.005),                                                # POT norm
        syst.SystematicList(
            [syst.SampleSystematic(list(dv[1:]), cvdf=dv[0]) for dv in detvars] +           # detector variations
            [syst.SampleSystematic(chi2_frames[V]) for V in calo_variations] +              # this model's base calo variations
            [syst.SampleSystematic(chi2_frames[V], norm=n) for (V, n) in smear_terms] +     # this model's smearing terms
            cosmic_systs +                                                                  # cosmic norm (SBND only)
            tracksplit_systs +                                                              # track splitting (ICARUS only)
            [syst.StatSampleSystematic(df)]                                                 # MC stat
        ),
        syst.SystSampleSystematic(df_ooav),                                                 # OOAV/dirt, 100% unc.
    ])

# WHICH MODELS EACH STAGE GETS
# The angle-inclusive stage and the angular quintile stages can carry different
# model sets (see SMEAR_MODELS_INCLUSIVE / _ANGULAR in the RUN CONFIGURATION
# cell). A stage is inclusive when it is its own base -- the angular stages are
# the ones STAGE_INFO tags with an angle.
def is_inclusive_stage(stage):
    return STAGE_INFO[stage].get("angle") is None


def stage_models(stage):
    """The smearing models plotted at `stage`."""
    return SMEAR_MODELS_INCLUSIVE if is_inclusive_stage(stage) else SMEAR_MODELS_ANGULAR


# Per-model, per-stage: the model's budget plus the trigger (flash-PE scale)
# SelectionSystematic for that stage. Only SelectionSystematic is
# stage-specific -- it names its universe columns up front; every other term
# is handed the cut column at cov() time, so the same objects serve all stages.
# These hold references to df (no copies), so the extra stages are near-free.
def model_systs_for(stage):
    return {
        tag: syst.SystematicList([
            model_systematics(terms, calo, g4_norm),
            syst.SelectionSystematic(df, ["%s_trig_up" % stage, "%s_trig_dn" % stage]),
        ])
        for (tag, label, terms, calo, g4_norm) in stage_models(stage)
    }

model_systs = {stage: model_systs_for(stage) for stage in ALL_STAGES}

## PID comparisons by particle type, per smearing model, per angular bin


**GUMPLE semantics note**: in the GUMPLE dataframes `mu_chi2_of_prot_cand` /
`prot_chi2_of_prot_cand` (and their variation columns) aggregate over **all** proton
candidates (min $\chi^2_\mu$ / max $\chi^2_p$), not the single proton candidate of the
reCAF production. The per-track equivalents `*_chi2_of_lead_prot` exist but have no
smearing-variation columns, so the plots below keep the aggregated variables --
distributions shift vs the -14 (reCAF) versions for events with more than two
candidate pfps. The muon candidate is also redefined (longest passing track, was
best $\chi^2$ ratio).

**Update (n_pfp == 2 cut)**: every cut stage now requires exactly one
proton candidate, so the aggregated `*_chi2_of_prot_cand` columns are
identical to the per-track values for all plotted events -- the
aggregation caveat above no longer applies to these comparisons.

**Chi2 flavor**: the four column names below are the flavor-free nominal ones.
`apply_flavor` has already rewritten them, on every frame, to hold the values of
the flavor selected by `GUMPLE_CHI2_FLAVOR` (see `CHI2_FLAVORS`), so what these
plots actually show is `*_chi2<plane><nominal>_of_*_cand` for that flavor.


In [ ]:
plotvars = [
    "mu_chi2_of_prot_cand",
    "prot_chi2_of_prot_cand",
    "mu_chi2_of_mu_cand",
    "prot_chi2_of_mu_cand",
]

bins = [
    np.linspace(0, 60, 13),
    np.linspace(0, 210, 15),  # 0-210, 14 bins of 15 (twice the bin count of the 7x30 version)
    np.linspace(0, 60, 13),
    np.linspace(0, 300, 21),
]

labels = [
    "Proton Cand. $\\chi^2_\\mu$",
    "Proton Cand. $\\chi^2_p$",
    "Muon Cand. $\\chi^2_\\mu$",
    "Muon Cand. $\\chi^2_p$",
]


In [ ]:
def inner(dat):
    (tag, stage, v, b, l) = dat
    bkdwn = breakdown_pdg_p if "prot_cand" in v else breakdown_pdg_mu
    # the plot title is the stage's base cut name; `stage` itself selects the rows
    return tag, stage, v, make_plot_data(v, b, stage, "glob_scale", bkdwn, AREANORM, pdg_labels,
                                         pdg_colors, l, STAGE_INFO[stage]["title"], fillna=-1,
                                         det=DETECTOR, syst=model_systs[stage][tag])


In [ ]:
# Area (shape) normalization only -- see the AREANORM note at the top. The full
# on-beam streams are loaded here, so an absolute-rate comparison must not be made.
assert AREANORM, "PID comparisons are area normalized only"

# cut stages x 4 variables x (1 inclusive + 2 angles x 5 quintiles) x the models
# that stage carries (stage_models: the inclusive stage and the angular ones can
# differ -- see SMEAR_MODELS_INCLUSIVE / _ANGULAR)
inputs = [(tag, stage, v, b, l)
          for (v, b, l) in zip(plotvars, bins, labels)
          for (cutf, base, _title, _tag) in CUT_STAGES
          for stage in stages_for(base, v)
          for (tag, label, terms, calo, g4_norm) in stage_models(stage)]
print("%d comparisons" % len(inputs))

# SBND loads 13 ~2.8 GB MC CV files (~36 GB) and 18x the on-beam data, so its CV
# frame is several times ICARUS's; each forked worker touches enough of it that 4
# workers put the machine under memory pressure. Halve the pool there.
NPROC = 2 if DETECTOR == "SBND" else 4

all_plotdata_pid = {}
with Pool(NPROC) as p:
    for tag, stage, v, plotdata in tqdm(p.imap_unordered(inner, inputs), total=len(inputs)):
        all_plotdata_pid[(tag, stage, v)] = plotdata

In [ ]:
ifig = 0
for v in plotvars:
    for (cutf, base, _title, _tag) in CUT_STAGES:
        for stage in stages_for(base, v):
            for (tag, label, terms, calo, g4_norm) in stage_models(stage):
                plt.figure(ifig)
                plotdata = all_plotdata_pid[(tag, stage, v)]
                fig, ax0, ax1 = ratio_plot(plt, plotdata)

                # mark the chi2 flavor and the smearing systematic model on the plot,
                # continuing the annotation stack ratio_plot started with the chi2
                # (see ANNOT_Y0 / ANNOT_DY)
                ax0.text(0.05, ANNOT_Y0 - ANNOT_DY, "%s $\\chi^2$" % CHI2_LABEL, verticalalignment="top",
                         horizontalalignment="left", fontsize=FONTSIZE-2, transform=ax0.transAxes)
                ax0.text(0.05, ANNOT_Y0 - 2*ANNOT_DY, label, verticalalignment="top", horizontalalignment="left",
                         fontsize=FONTSIZE-2, transform=ax0.transAxes)

                # and, for an angular bin, the angular cut it was made under
                info = STAGE_INFO[stage]
                if info["label"] is not None:
                    ax0.text(0.05, ANNOT_Y0 - 3*ANNOT_DY, info["label"], verticalalignment="top",
                             horizontalalignment="left", fontsize=FONTSIZE-2,
                             transform=ax0.transAxes)

                if DOSAVE:
                    # the flavor is already the sub-directory; it is repeated in the
                    # filename so the plots stay unique if ever pooled into one place
                    savename = "%s_%s_%s_%s_bkdwnpdg_%s%s" % (
                        plotdata["det"].replace(" ", "-"), CHI2_FLAVOR, info["cuttag"], v,
                        tag, info["tag"])
                    plt.savefig(PLOTDIR + "/pdf/" + savename + ".pdf", bbox_inches="tight")
                    plt.savefig(PLOTDIR + "/png/" + savename + ".png", bbox_inches="tight")
                    plt.close()
                else:
                    ifig += 1